# SiPM data check

Doesn't train anything — just makes sure the data actually loads right in Colab.
Mounts Drive, reads the label JSONs, runs the loader, and draws the red/blue masks
on a few chips so I can see they line up.

Shift+Enter runs a cell (same as MATLAB).

Drive needs a folder called `sipm` with the label `.json` files, the `cropped/` folder
(per-tray subfolders inside), and `sipm_dataset.py`.

Set the GPU now even though this part doesn't need it: Runtime → Change runtime type → GPU.

mount drive — click through the popup it gives you

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

point at the sipm folder. change this line if it's somewhere else

In [ ]:
import os
ROOT = '/content/drive/MyDrive/sipm'   # <-- change if your folder is elsewhere
assert os.path.isdir(ROOT), f"Can't find {ROOT}. Check the folder name/location in Drive."
print("Contents of your sipm folder:")
print(os.listdir(ROOT))

torch is already in colab, just need python to see sipm_dataset.py

In [ ]:
import sys
sys.path.append(ROOT)
from sipm_dataset import load_records, SiPMDataset
print("loader imported OK")

read the labels

In [ ]:
import glob
json_files = sorted(glob.glob(os.path.join(ROOT, '*.json')))
print("label files found:", [os.path.basename(f) for f in json_files])

records = load_records(json_files)
print(f"\ntotal chips: {len(records)}")
dmg = sum(1 for r in records if r['status']=='damaged')
print(f"damaged: {dmg} | pristine: {len(records)-dmg}")

build the dataset. image_dir has to be the folder *containing* the tray folders

In [ ]:
ds = SiPMDataset(records, image_dir=os.path.join(ROOT, 'cropped'), size=512)
print(f"dataset ready: {len(ds)} chips")

# grab one chip to confirm shapes
x, y, has_damage = ds[0]
print("image tensor:", tuple(x.shape), "| label map:", tuple(y.shape),
      "| classes present:", sorted(set(y.flatten().tolist())), "| has_damage:", has_damage)

the actual check — draw the masks on and look at them.
red = real damage, blue = artifact. if the colors sit on the real scratches/bubbles/hairs
then the masks decoded and resized correctly and nothing's shifted.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from PIL import Image
from sipm_dataset import SiPMDataset as DS

# find a few damaged chips to display
dmg_idx = [i for i,r in enumerate(records) if r['status']=='damaged'][:6]

fig, axes = plt.subplots(2, 3, figsize=(13, 9))
for ax, i in zip(axes.flat, dmg_idx):
    r = records[i]
    x, y, _ = ds[i]
    ylab = y.numpy()
    # rebuild the displayed image (un-normalized) the same way the loader made it
    img = DS._pad_square(Image.open(os.path.join(ROOT,'cropped',r['key'])).convert('RGB'),0).resize((512,512))
    ov = np.array(img).astype(float)
    ov[ylab==1] = 0.5*ov[ylab==1] + 0.5*np.array([255,0,0])   # damage -> red
    ov[ylab==2] = 0.5*ov[ylab==2] + 0.5*np.array([0,0,255])   # artifact -> blue
    ax.imshow(ov.astype('uint8'))
    ax.set_title(r['key']+f"  grade {r['grade']}", fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()
print("If the red/blue lands on the real defects, your data pipeline works. ✅")

---
if the overlays look right the data's fine and I can move on to training.
nothing else happens in this notebook.